# StandUp4AI Training - Self Contained

XLM-RoBERTa-base for word-level laughter detection.

## Features
- ✅ Self-contained: fetches data from GitHub
- ✅ Fixed AdamW import
- ✅ GPU enabled
- ✅ Auto-saves model to Google Drive

## Dataset
- French: 1,052 words
- English: 1,865 words
- Spanish: 286 words
- Total: 3,203 words

In [ ]:
# Install dependencies
!pip install -q torch transformers scikit-learn pandas numpy requests

In [ ]:
# Import libraries (FIXED: AdamW from torch.optim)
import torch
import pandas as pd
import numpy as np
import requests
import io
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW  # FIXED: from torch.optim
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# Config
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 64
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/standup4ai_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")

## Fetch Data from GitHub

In [ ]:
# Fetch CSV data from GitHub raw URLs
GITHUB_BASE = "https://raw.githubusercontent.com/Das-rebel/standup4ai-colab/main"

files = {
    '-1FrUOEswOk.csv': 'fr',
    '0g7nezWZyfY.csv': 'en',
    '1xvwYZwm8Ig.csv': 'en',
    '6JQzl2LlXbQ.csv': 'es'
}

dfs = []
for fname, lang in files.items():
    url = f"{GITHUB_BASE}/{fname}"
    print(f"Fetching {fname}...")
    r = requests.get(url)
    df = pd.read_csv(io.StringIO(r.text))
    df['language'] = lang
    dfs.append(df)
    print(f"  -> {len(df)} rows")

data = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(data)} samples")

## Preprocess

In [ ]:
# Convert labels: L -> 1, O -> 0
data['label'] = data['label'].map({'L': 1, 'O': 0})
print(f"NaN labels: {data['label'].isna().sum()}")
print(f"\nLabel dist:\n{data['label'].value_counts()}")

# Stratified split by language
train_dfs, val_dfs = [], []
for lang in data['language'].unique():
    ld = data[data['language'] == lang]
    tr, vl = train_test_split(ld, test_size=0.2, random_state=42, stratify=ld['label'])
    train_dfs.append(tr)
    val_dfs.append(vl)
    print(f"{lang}: train={len(tr)}, val={len(vl)}")

train_data = pd.concat(train_dfs)
val_data = pd.concat(val_dfs)
print(f"\nTotal: train={len(train_data)}, val={len(val_data)}")

## Dataset & Model

In [ ]:
class LaughterDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), add_special_tokens=True, max_length=self.max_len,
                           padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

class XLMRClassifier(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Linear(self.xlmr.config.hidden_size, 2)
    def forward(self, input_ids, attention_mask):
        out = self.xlmr(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0])
        return self.classifier(cls)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create datasets & dataloaders
train_ds = LaughterDataset(train_data['text'].values, train_data['label'].values, tokenizer, MAX_LEN)
val_ds = LaughterDataset(val_data['text'].values, val_data['label'].values, tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Init model
model = XLMRClassifier(MODEL_NAME).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*len(train_loader)*EPOCHS), num_training_steps=len(train_loader)*EPOCHS)
print("Model ready!")

## Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, preds, labels = 0, [], []
    criterion = torch.nn.CrossEntropyLoss()
    for batch in loader:
        optimizer.zero_grad()
        logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        loss = criterion(logits, batch['label'].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds.extend(torch.argmax(logits, 1).cpu().numpy())
        labels.extend(batch['label'].numpy())
    return total_loss / len(loader), f1_score(labels, preds, average='weighted')

def eval_epoch(model, loader):
    model.eval()
    total_loss, preds, labels = 0, [], []
    criterion = torch.nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            loss = criterion(logits, batch['label'].to(DEVICE))
            total_loss += loss.item()
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labels.extend(batch['label'].numpy())
    return total_loss / len(loader), f1_score(labels, preds, average='weighted'), preds, labels

best_f1 = 0
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}

print("Training...")
for epoch in range(EPOCHS):
    tl, tf1 = train_epoch(model, train_loader, optimizer, scheduler)
    vl, vf1, _, _ = eval_epoch(model, val_loader)
    history['train_loss'].append(tl)
    history['train_f1'].append(tf1)
    history['val_loss'].append(vl)
    history['val_f1'].append(vf1)
    print(f"Epoch {epoch+1}/{EPOCHS}: train_loss={tl:.4f}, train_f1={tf1:.4f}, val_loss={vl:.4f}, val_f1={vf1:.4f}")
    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")
        print(f"  -> Saved! (F1: {best_f1:.4f})")

print(f"\nBest Val F1: {best_f1:.4f}")

## Per-Language Evaluation

In [ ]:
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model.pt"))
_, _, predictions, actual = eval_epoch(model, val_loader)

val_data_eval = val_data.copy()
val_data_eval['pred'] = predictions

print("="*50)
print("PER-LANGUAGE RESULTS")
print("="*50)

results = {}
for lang in ['fr', 'en', 'es']:
    ld = val_data_eval[val_data_eval['language'] == lang]
    if len(ld) > 0:
        f1 = f1_score(ld['label'], ld['pred'], average='weighted')
        results[lang] = f1
        status = "PASS" if f1 >= 0.70 else "FAIL"
        print(f"{lang.upper()}: F1={f1:.4f} [{status}] (n={len(ld)})")

overall = f1_score(actual, predictions, average='weighted')
status = "PASS" if overall >= 0.70 else "FAIL"
print(f"\nOVERALL: F1={overall:.4f} [{status}]")

## Save Model & Metrics

In [ ]:
import json

# Save model
model_path = f"{OUTPUT_DIR}/xlmr_laughter"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# Save history
with open(f"{OUTPUT_DIR}/history.json", 'w') as f:
    json.dump(history, f)

# Save results
with open(f"{OUTPUT_DIR}/results.json", 'w') as f:
    json.dump({'overall_f1': overall, 'per_lang': results, 'best_f1': best_f1}, f)

print("Saved!")
print(f"\nFiles in {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    print(f"  {f}")

## Done! ✅

Model saved to `/content/drive/MyDrive/standup4ai_models/`

**Target:** French, English, Spanish F1 >= 0.70